## Environment Setup and Library Imports

### 1. Installing Dependencies
The project requires the `jiwer` library for calculating **Word Error Rate (WER)** and **Character Error Rate (CER)**, which are essential metrics for evaluating the performance of lipreading models.

```python
!pip install -q jiwer


In [ ]:
# Install jiwer for calculating Word Error Rate (WER) and Character Error Rate (CER)
print("### Installing dependencies...")
!pip install -q jiwer

# Import all required libraries
print("### Importing libraries...")
import os
import cv2
import csv
import glob
import jiwer
import numpy as np
import tensorflow as tf
from typing import List
from matplotlib import pyplot as plt

from google.colab import drive
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, Conv3D, LSTM, Dense, Dropout, Bidirectional, MaxPool3D,
                                     Activation, Reshape, TimeDistributed) # <-- 'Input' was added here
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import LearningRateScheduler, Callback

print("### Setup complete.")

## GPU Configuration and Mixed Precision Setup

### Purpose
This section configures the training environment to **leverage GPU acceleration** and enables **mixed precision training** for optimal performance.

### Key Steps
1. **GPU Detection**:  
   The code checks if a physical GPU is available using:
   ```python
   tf.config.list_physical_devices('GPU')


In [ ]:
# Configure GPU for optimal performance with mixed precision
print("### Configuring GPU for optimal performance...")
if tf.config.list_physical_devices('GPU'):
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy('mixed_float16')
    print("Mixed precision enabled for GPU.")
else:
    print("No GPU found. Running in standard precision.")

## Google Drive Setup and Configuration

### Overview
This section configures **Google Drive paths** and **training parameters** for running the LipNet model on **Google Colab**. It ensures results, checkpoints, and logs are stored persistently in Drive while allowing faster training with a **local copy of the dataset** on the Colab VM.

---

### 1. Google Drive Mounting
- Mounts the user's Google Drive to `/content/drive`.
- Allows seamless access to datasets, checkpoints, and logs stored in Google Drive.

```python
drive.mount('/content/drive')
```

### 2. Project Folder Structure
- **GDRIVE_PROJECT_FOLDER**: Main folder in Google Drive where model checkpoints, graphs, and logs will be saved.
- **GDRIVE_DATA_PATH**: Path to the preprocessed dataset stored in Google Drive.
- **LOCAL_DATA_PATH**: Local cache path in Colab VM for faster I/O during training.

The script automatically creates required subfolders if they do not exist:
- `checkpoints/` → Model checkpoints saved during training.
- `graphs/` → Training/validation loss and accuracy plots.
- `training_log.csv` → Detailed log file tracking metrics across epochs.

### 3. Training Configuration
Key parameters defined for training:

- **VOCAB**: The vocabulary used for lip-reading predictions, including:
  - Lowercase letters a–z
  - Apostrophe `'`, question mark `?`, exclamation `!`
  - Digits 1–9
  - Space character
- **BATCH_SIZE = 2**: Small batch size chosen due to Colab GPU memory limitations.
- **EPOCHS = 100**: Maximum number of training epochs.

### 4. Colab Workflow
- Data is stored in Google Drive but copied to local storage (`/content/s1_processed_local`) for speed.
- Training checkpoints and logs are saved back to Google Drive to avoid loss when the Colab session disconnects.
- This setup supports resuming training from the latest checkpoint if the session is interrupted.

In [ ]:
print("### Configuring paths and mounting Google Drive...")
# Mount Google Drive
drive.mount('/content/drive')

### --- SETUP PATHS --- ###

#Set the FOLDER where results will be SAVED on Google Drive
GDRIVE_PROJECT_FOLDER = '/content/drive/MyDrive/VisoSpeak_Project_A100'

#Set the FOLDER where your original dataset is STORED on Google Drive
GDRIVE_DATA_PATH = '/content/drive/MyDrive/s1_processed'

#FOLDER for the FAST LOCAL COPY of the data on the Colab machine
LOCAL_DATA_PATH = '/content/s1_processed_local'

# --- Automatic Path Generation ---
# IMPORTANT: Checkpoints, graphs, and logs will be saved directly to your Google Drive
CHECKPOINT_DIR = os.path.join(GDRIVE_PROJECT_FOLDER, 'checkpoints')
GRAPHS_DIR = os.path.join(GDRIVE_PROJECT_FOLDER, 'graphs')
LOGS_PATH = os.path.join(GDRIVE_PROJECT_FOLDER, 'training_log.csv')

# Create project directories on Google Drive if they don't exist
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(GRAPHS_DIR, exist_ok=True)
print(f"Results will be saved to: {GDRIVE_PROJECT_FOLDER}")
print(f"Training data will be read from: {LOCAL_DATA_PATH} (after copy)")

# --- Model & Training Parameters ---
VOCAB = [x for x in "abcdefghijklmnopqrstuvwxyz'?!123456789 "]
BATCH_SIZE = 2
EPOCHS = 100

## Data Verification and Cleanup

### Overview
This section **verifies dataset integrity** and **copies clean data** to local storage for faster training. It ensures that every video file has a corresponding alignment file and vice versa before proceeding with the training process.

---

### 1. Dataset Integrity Check
The script performs a comprehensive verification of the dataset by:
- **Scanning video files**: Locates all `.mpg` files in the Google Drive data directory
- **Scanning alignment files**: Locates all `.align` files in the `align/` subdirectory
- **Cross-referencing**: Compares basenames to identify any orphaned files

### 2. File Matching Logic
```python
# Get the base filenames
video_files = glob.glob(os.path.join(GDRIVE_DATA_PATH, '*.mpg'))
align_files = glob.glob(os.path.join(GDRIVE_ALIGN_PATH, '*.align'))

video_basenames = {os.path.splitext(os.path.basename(f))[0] for f in video_files}
align_basenames = {os.path.splitext(os.path.basename(f))[0] for f in align_files}

# Find any files that don't have a matching pair
videos_without_aligns = video_basenames - align_basenames
aligns_without_videos = alignment_basenames - video_basenames
```

### 3. Verification Outcomes

####  **Success Case**: Clean Dataset
- All video files have matching alignment files
- Data is automatically copied to local storage using `rsync`
- Training can proceed immediately

####  **Failure Case**: Dataset Mismatch
When orphaned files are detected, the script provides:
- **Detailed report** of mismatched files
- **Automated cleanup commands** (`!rm` statements)
- **Step-by-step instructions** for manual cleanup

### 4. Cleanup Process
If dataset issues are found:

1. **Review the mismatch report** showing orphaned files
2. **Copy the generated `!rm` commands** into a new code cell
3. **Execute the cleanup commands** to remove orphaned files from Google Drive
4. **Re-run the verification cell** to confirm the dataset is clean

### 5. Data Copy Operation
Once verification passes:
- **Source**: Google Drive data directory (`GDRIVE_DATA_PATH`)
- **Destination**: Local Colab storage (`LOCAL_DATA_PATH`)
- **Method**: `rsync` with progress reporting for efficient transfer
- **Purpose**: Faster I/O during training while maintaining Google Drive backup

### 6. Error Handling
The script uses **exception-based flow control**:
- Raises an exception if cleanup is required
- Prevents training from starting with corrupted data
- Forces manual intervention to ensure data quality

In [ ]:
### --- Verify, Clean, and Copy Data --- ###
print("### Verifying dataset integrity on Google Drive...")

GDRIVE_ALIGN_PATH = os.path.join(GDRIVE_DATA_PATH, 'align')

# Get the base filenames
video_files = glob.glob(os.path.join(GDRIVE_DATA_PATH, '*.mpg'))
align_files = glob.glob(os.path.join(GDRIVE_ALIGN_PATH, '*.align'))

video_basenames = {os.path.splitext(os.path.basename(f))[0] for f in video_files}
align_basenames = {os.path.splitext(os.path.basename(f))[0] for f in align_files}

# Find any files that don't have a matching pair
videos_without_aligns = video_basenames - align_basenames
aligns_without_videos = align_basenames - video_basenames

# --- REPORT AND PROVIDE CLEANUP COMMANDS ---
if not videos_without_aligns and not aligns_without_videos:
    print("✅ Dataset integrity check passed. All files have a matching pair.")

    # ---  COPY CLEAN DATA TO LOCAL DISK --- ###
    print(f"\n### Copying verified data from {GDRIVE_DATA_PATH} to {LOCAL_DATA_PATH}...")
    !rsync -a --info=progress2 {GDRIVE_DATA_PATH}/ {LOCAL_DATA_PATH}
    print("\n### Data copy complete. You may now proceed. ###")

else:
    print("\n❌ MISMATCH FOUND IN DATASET! Please clean your data.")
    if videos_without_aligns:
        print("\n--- The following VIDEOS have no matching .align file: ---")
        for basename in sorted(list(videos_without_aligns)):
            print(f'!rm "{os.path.join(GDRIVE_DATA_PATH, basename)}.mpg"')

    if aligns_without_videos:
        print("\n--- The following ALIGNMENTS have no matching .mpg file: ---")
        for basename in sorted(list(aligns_without_videos)):
            print(f'!rm "{os.path.join(GDRIVE_ALIGN_PATH, basename)}.align"')

    print("\n--- ACTION REQUIRED ---")
    print("1. Create a new code cell.")
    print("2. Copy and paste the `!rm` commands for the orphan files into the new cell.")
    print("3. Run the cell to delete the files from your Google Drive.")
    print("4. After deleting, re-run this verification cell to confirm the dataset is clean.")
    raise Exception("Dataset cleanup required before training can continue.")

## Preprocessing Functions and Character Maps

### Overview
This section defines the **preprocessing pipeline** for converting raw video and text data into tensors suitable for LipNet training.

---

### 1. Character Vocabulary Mapping
- **`char_to_num`**: Converts characters to numerical indices
- **`num_to_char`**: Converts predictions back to readable text
- Uses the predefined `VOCAB` for consistent encoding/decoding

### 2. Video Processing (`load_video`)
- Extracts frames from `.mpg` files
- Converts to grayscale and crops to lip region `[190:236, 80:220]`
- Applies normalization: `(frames - mean) / std`

### 3. Alignment Processing (`load_alignments`) 
- Reads `.align` files and filters out silence tokens
- Splits text into individual characters
- Converts characters to numerical indices

### 4. Data Loading (`load_data`)
- Combines video and alignment loading
- Handles different path input types (string, bytes, tensor)
- Returns processed video frames and encoded text

### 5. TensorFlow Integration (`mappable_function`)
- Wraps the data loading function for use in `tf.data` pipelines
- Ensures compatibility with TensorFlow's graph execution

In [ ]:
print("### Defining preprocessing functions and character maps...")

# --- Character to Number Mapping ---
char_to_num = tf.keras.layers.StringLookup(vocabulary=VOCAB, oov_token="")
num_to_char = tf.keras.layers.StringLookup(
    vocabulary=char_to_num.get_vocabulary(), oov_token="", invert=True
)
print(f"Vocabulary size: {char_to_num.vocabulary_size()}")


# --- Data Loading Functions ---
def load_video(path:str):
    cap = cv2.VideoCapture(path)
    frames = []
    for _ in range(int(cap.get(cv2.CAP_PROP_FRAME_COUNT))):
        ret, frame = cap.read()
        if not ret: break
        frame = tf.image.rgb_to_grayscale(frame)
        frames.append(frame[190:236, 80:220, :])
    cap.release()
    if not frames: return np.array([])
    mean = tf.math.reduce_mean(frames)
    std = tf.math.reduce_std(tf.cast(frames, tf.float32))
    return tf.cast((frames - mean), tf.float32) / std

def load_alignments(path:str):
    with open(path, 'r') as f: lines = f.readlines()
    tokens = []
    for line in lines:
        line = line.split()
        if line[2] != 'sil': tokens.extend([' ', line[2]])
    return char_to_num(tf.reshape(tf.strings.unicode_split(tokens, input_encoding='UTF-8'), (-1)))[1:]

def load_data(path):
    if hasattr(path, 'numpy'):
        path_str = path.numpy().decode('utf-8')
    elif isinstance(path, bytes):
        path_str = path.decode('utf-8')
    else:
        path_str = path

    base_dir = os.path.dirname(path_str)
    file_name = os.path.basename(path_str).split('.')[0]
    alignment_path = os.path.join(base_dir, 'align', f'{file_name}.align')

    frames = load_video(path_str)
    alignments = load_alignments(alignment_path)
    return frames, alignments

@tf.function
def mappable_function(path:str):
    return tf.py_function(load_data, [path], (tf.float32, tf.int64))

## TensorFlow Data Pipeline

### Overview
This section builds an efficient **tf.data pipeline** for loading, processing, and batching the video dataset for training.

---

### 1. Dataset Creation
```python
data = tf.data.Dataset.list_files(os.path.join(GDRIVE_DATA_PATH, '*.mpg'))
data = data.shuffle(1000, reshuffle_each_iteration=False)
data = data.map(mappable_function)
```
- **File listing**: Discovers all `.mpg` video files in the dataset
- **Shuffling**: Randomizes order with buffer size of 1000 files
- **Mapping**: Applies preprocessing functions to load video frames and alignments

### 2. Batching and Optimization
```python
data = data.padded_batch(BATCH_SIZE, padded_shapes=([75, None, None, None], [40]))
data = data.prefetch(tf.data.AUTOTUNE)
```
- **Padded batching**: Groups samples with padding to handle variable sequence lengths
  - Video shape: `[75, None, None, None]` (max 75 frames, variable height/width)
  - Text shape: `[40]` (max 40 characters)
- **Prefetching**: Loads next batch while current batch is being processed

### 3. Train/Validation Split
```python
total_size = tf.data.experimental.cardinality(data).numpy()
train_size = max(1, int(0.9 * total_size))
test_data = data.skip(train_size)
train_data = data.take(train_size)
```
- **90/10 split**: 90% for training, 10% for validation
- **Sequential split**: Training takes first portion, validation takes remainder
- Ensures at least 1 batch for training even with very small datasets

In [ ]:
print("### Building and testing the tf.data pipeline...")

# lists files from the fast LOCAL_DATA_PATH
data = tf.data.Dataset.list_files(os.path.join(GDRIVE_DATA_PATH, '*.mpg'))
data = data.shuffle(1000, reshuffle_each_iteration=False)
data = data.map(mappable_function)

data = data.padded_batch(BATCH_SIZE, padded_shapes=([75, None, None, None], [40]))
data = data.prefetch(tf.data.AUTOTUNE)

# Split the data into training and validation sets
total_size = tf.data.experimental.cardinality(data).numpy()
train_size = max(1, int(0.9 * total_size))
test_data = data.skip(train_size)
train_data = data.take(train_size)

print(f"Dataset ready. Total batches: {total_size}, Training batches: {train_size}, Validation batches: {max(0, total_size - train_size)}")

## LipNet Model Architecture

### Overview
This section defines the **LipNet neural network architecture** for lip-reading, combining 3D CNNs for spatial-temporal feature extraction with bidirectional LSTMs for sequence modeling.

---

### 1. Input Layer
```python
Input(shape=(None, 46, 140, 1), name='input_layer')
```
- **Flexible sequence length**: `None` allows variable number of frames
- **Frame dimensions**: 46×140 pixels (lip region after cropping)
- **Grayscale**: Single channel input

### 2. 3D Convolutional Layers
```python
Conv3D(128, 3, padding='same') → ReLU → MaxPool3D((1, 2, 2))
Conv3D(256, 3, padding='same') → ReLU → MaxPool3D((1, 2, 2))
Conv3D(75, 3, padding='same') → ReLU → MaxPool3D((1, 2, 2))
```
- **Spatial-temporal features**: 3D convolutions capture lip movement patterns
- **Progressive filters**: 128 → 256 → 75 channels
- **Pooling**: Reduces spatial dimensions while preserving temporal information

### 3. Feature Flattening
```python
TimeDistributed(Reshape((5 * 17 * 75,)), name='reshape_per_frame')
TimeDistributed(Dense(128, activation='relu'), name='timedist_dense')
```
- **Per-frame processing**: Flattens each frame's features independently
- **Dense compression**: Reduces feature dimensions to 128 per frame

### 4. Sequence Modeling
```python
Bidirectional(LSTM(128, return_sequences=True)) + Dropout(0.5)
Bidirectional(LSTM(128, return_sequences=True)) + Dropout(0.5)
```
- **Bidirectional LSTMs**: Process sequences forward and backward
- **Stacked layers**: Two LSTM layers for complex pattern learning
- **Dropout**: 50% dropout for regularization

### 5. Output Layer
```python
Dense(char_to_num.vocabulary_size() + 1, activation='softmax')
```
- **Character prediction**: Outputs probability for each character in vocabulary
- **CTC compatibility**: Extra dimension (+1) for blank token in CTC loss

In [ ]:
print("### Building the LipNet model architecture...")

def build_model():
    """Builds the LipNet-style sequential model with a flexible input shape."""
    model = Sequential([
        Input(shape=(None, 46, 140, 1), name='input_layer'),
        Conv3D(128, 3, padding='same', name='conv3d_1'),
        Activation('relu', name='relu_1'),
        MaxPool3D((1, 2, 2), name='maxpool_1'),
        Conv3D(256, 3, padding='same', name='conv3d_2'),
        Activation('relu', name='relu_2'),
        MaxPool3D((1, 2, 2), name='maxpool_2'),
        Conv3D(75, 3, padding='same', name='conv3d_3'),
        Activation('relu', name='relu_3'),
        MaxPool3D((1, 2,2), name='maxpool_3'),
        TimeDistributed(Reshape((5 * 17 * 75,)), name='reshape_per_frame'),
        TimeDistributed(Dense(128, activation='relu'), name='timedist_dense'),
        Bidirectional(LSTM(128, kernel_initializer='Orthogonal', return_sequences=True), name='bidi_lstm_1'),
        Dropout(0.5),
        Bidirectional(LSTM(128, kernel_initializer='Orthogonal', return_sequences=True), name='bidi_lstm_2'),
        Dropout(0.5),
        Dense(char_to_num.vocabulary_size() + 1, kernel_initializer='he_normal', activation='softmax', dtype='float32', name='output_dense')
    ])
    return model

## Loss Function and Custom Callbacks

### Overview
This section defines the **CTC loss function** for sequence alignment and **custom callbacks** for monitoring training progress, saving checkpoints, and generating visualization plots.

---

### 1. CTC Loss Function
```python
def CTCLoss(y_true, y_pred):
    batch_len = tf.cast(tf.shape(y_true)[0], dtype="int64")
    input_length = tf.cast(tf.shape(y_pred)[1], dtype="int64")
    label_length = tf.cast(tf.shape(y_true)[1], dtype="int64")
    loss = tf.keras.backend.ctc_batch_cost(y_true, y_pred, input_length, label_length)
    return loss
```
- **CTC (Connectionist Temporal Classification)**: Handles variable-length sequence alignment
- **No explicit alignment needed**: Automatically aligns predictions with ground truth
- **Batch processing**: Computes loss for entire batches efficiently

### 2. Learning Rate Scheduler
```python
def learning_rate_scheduler(epoch, lr):
    if epoch < 30:
        return lr
    else:
        return lr * tf.math.exp(-0.1).numpy()
```
- **Static phase**: Maintains original learning rate for first 30 epochs
- **Decay phase**: Exponentially reduces learning rate after epoch 30
- **Gradual convergence**: Helps fine-tune model parameters in later stages

### 3. Custom Metrics & Plots Callback
The `MetricsAndPlotsCallback` class provides comprehensive training monitoring:

#### Initialization
- Sets up CSV logging with headers: epoch, loss, val_loss, CER, WER, learning_rate
- Configures checkpoint and visualization directories
- Defines checkpoint saving frequency (`save_every_n_epochs=5`)

#### Per-Epoch Processing
```python
def on_epoch_end(self, epoch, logs=None):
    # CTC decoding with beam search
    decoded = tf.keras.backend.ctc_decode(yhat, input_length, beam_width=10)
    
    # Calculate WER and CER metrics
    wer = jiwer.wer(ground_truths, predictions)
    cer = jiwer.cer(ground_truths, predictions)
```

**Key features:**
- **CTC Decoding**: Uses beam search (width=10) for better predictions
- **Text Processing**: Normalizes text with lowercase and space removal
- **Error Metrics**: Calculates Word Error Rate (WER) and Character Error Rate (CER)
- **CSV Logging**: Records all metrics for analysis
- **Checkpoint Saving**: Saves model weights every N epochs

#### Prediction Visualization
```python
def save_prediction_plot(self, data, predictions, epoch):
    plt.figure(figsize=(10, 6))
    plt.imshow(tf.squeeze(frame))
    plt.title(f"Epoch: {epoch}\\nOriginal: {original_text}\\nPrediction: {predicted_text}")
```
- **Visual monitoring**: Shows middle frame from video sequence
- **Text comparison**: Displays original vs predicted text
- **Progress tracking**: Saves plots every checkpoint epoch

In [ ]:
print("### Defining loss function and custom callbacks...")

def CTCLoss(y_true, y_pred):
    batch_len = tf.cast(tf.shape(y_true)[0], dtype="int64")
    input_length = tf.cast(tf.shape(y_pred)[1], dtype="int64")
    label_length = tf.cast(tf.shape(y_true)[1], dtype="int64")
    input_length = input_length * tf.ones(shape=(batch_len, 1), dtype="int64")
    label_length = label_length * tf.ones(shape=(batch_len, 1), dtype="int64")
    loss = tf.keras.backend.ctc_batch_cost(y_true, y_pred, input_length, label_length)
    return loss

def learning_rate_scheduler(epoch, lr):
    if epoch < 30:
        return lr
    else:
        # The .numpy() converts the result to a standard number
        return lr * tf.math.exp(-0.1).numpy()

class MetricsAndPlotsCallback(Callback):
    """Callback to log metrics, save plots, and save checkpoints every N epochs."""
    def __init__(self, test_dataset, logs_path, graphs_dir, checkpoint_dir, save_every_n_epochs=5):
        super().__init__()
        self.test_dataset = test_dataset
        self.logs_path = logs_path
        self.graphs_dir = graphs_dir
        self.checkpoint_dir = checkpoint_dir
        self.save_every_n_epochs = save_every_n_epochs
        if not os.path.exists(self.logs_path) or os.path.getsize(self.logs_path) == 0:
            with open(self.logs_path, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['epoch', 'loss', 'val_loss', 'cer', 'wer', 'learning_rate'])

    def on_epoch_end(self, epoch, logs=None):
        data = next(iter(self.test_dataset))
        yhat = self.model.predict(data[0], verbose=0)
        input_length = [yhat.shape[1]] * yhat.shape[0]
        decoded = tf.keras.backend.ctc_decode(yhat, input_length, greedy=False, beam_width=10)[0][0].numpy()

        ground_truths = [jiwer.RemoveMultipleSpaces()(jiwer.ToLowerCase()(tf.strings.reduce_join(num_to_char(d)).numpy().decode('utf-8').strip())) for d in data[1]]
        predictions = [jiwer.RemoveMultipleSpaces()(jiwer.ToLowerCase()(tf.strings.reduce_join(num_to_char(d)).numpy().decode('utf-8').strip())) for d in decoded]
        wer, cer = (jiwer.wer(ground_truths, predictions), jiwer.cer(ground_truths, predictions))
        logs['wer'], logs['cer'] = wer, cer

        lr = self.model.optimizer.learning_rate.numpy()

        with open(self.logs_path, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([epoch + 1, logs.get('loss'), logs.get('val_loss'), cer, wer, lr])
        print(f"\nEpoch {epoch+1} - Val CER: {cer:.4f}, Val WER: {wer:.4f}")

        if (epoch + 1) % self.save_every_n_epochs == 0:
            checkpoint_path = os.path.join(self.checkpoint_dir, f'epoch_{epoch+1:03d}.weights.h5')
            self.model.save_weights(checkpoint_path)
            print(f"Epoch {epoch+1}: saving model to {checkpoint_path}")
            self.save_prediction_plot(data, predictions, epoch + 1)

    def save_prediction_plot(self, data, predictions, epoch):
        frame_idx = data[0][0].shape[0] // 2
        frame = data[0][0][frame_idx]
        original_text = tf.strings.reduce_join(num_to_char(data[1][0])).numpy().decode('utf-8')
        predicted_text = predictions[0]
        plt.figure(figsize=(10, 6))
        plt.imshow(tf.squeeze(frame))
        plt.title(f"Epoch: {epoch}\nOriginal: {original_text}\nPrediction: {predicted_text}", fontsize=12)
        plt.axis('off')
        save_path = os.path.join(self.graphs_dir, f'prediction_epoch_{epoch:03d}.png')
        plt.savefig(save_path)
        plt.close()
        print(f"Saved prediction example plot to {save_path}")

## Model Training Setup

### Overview
This section **compiles the model**, **handles checkpoint resumption**, and **starts the training process** with custom callbacks for monitoring and saving progress.

---

### 1. Model Compilation
```python
model = build_model()
model.compile(optimizer=Adam(learning_rate=0.0001), loss=CTCLoss)
```
- **Adam optimizer**: Learning rate of 0.0001 for stable convergence
- **CTC loss**: Custom loss function for sequence alignment
- **Model summary**: Displays architecture details and parameter count

### 2. Checkpoint Resumption
```python
checkpoint_files = glob.glob(os.path.join(CHECKPOINT_DIR, "epoch_*.weights.h5"))
latest_checkpoint_path = max(checkpoint_files)
initial_epoch = int(os.path.basename(latest_checkpoint_path).split('_')[-1].split('.')[0])
```
- **Automatic detection**: Finds latest checkpoint file in the checkpoint directory
- **Weight loading**: Restores model weights from the most recent save
- **Epoch tracking**: Extracts epoch number from filename to resume training correctly
- **Fallback handling**: Starts from scratch if no checkpoints are found

### 3. Callback Configuration
```python
schedule_callback = LearningRateScheduler(learning_rate_scheduler)
metrics_callback = MetricsAndPlotsCallback(
    test_dataset=test_data,
    logs_path=LOGS_PATH,
    graphs_dir=GRAPHS_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    save_every_n_epochs=5
)
```
- **Learning rate scheduling**: Applies exponential decay after epoch 30
- **Metrics tracking**: Calculates CER/WER and saves training logs
- **Checkpoint saving**: Automatically saves model weights every 5 epochs
- **Visualization**: Generates prediction plots for monitoring progress

### 4. Training Execution
```python
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=EPOCHS,
    initial_epoch=initial_epoch,
    callbacks=[schedule_callback, metrics_callback]
)
```
- **Dataset splitting**: Uses 90% for training, 10% for validation
- **Resume capability**: Continues from the last saved epoch
- **Progress monitoring**: Tracks loss, CER, WER, and learning rate throughout training

In [ ]:
### Compiling model and setting up for training...
model = build_model()
model.summary()
model.compile(optimizer=Adam(learning_rate=0.0001), loss=CTCLoss)
# --- Load latest checkpoint if it exists ---
initial_epoch = 0
checkpoint_files = glob.glob(os.path.join(CHECKPOINT_DIR, "epoch_*.weights.h5"))
latest_checkpoint_path = None

if checkpoint_files:
    latest_checkpoint_path = max(checkpoint_files)
    print(f"Resuming training from: {latest_checkpoint_path}")
    model.load_weights(latest_checkpoint_path)
    try:
        initial_epoch = int(os.path.basename(latest_checkpoint_path).split('_')[-1].split('.')[0])
        print(f"Starting from epoch {initial_epoch}")
    except:
        print("Could not determine epoch from filename, resuming from latest.")
else:
    print("No checkpoint found. Starting training from scratch.")

# --- Setup Callbacks ---
schedule_callback = LearningRateScheduler(learning_rate_scheduler)
metrics_callback = MetricsAndPlotsCallback(
    test_dataset=test_data,
    logs_path=LOGS_PATH,
    graphs_dir=GRAPHS_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    save_every_n_epochs=5
)

# --- Train the model ---
print(f"\n### Starting training for {EPOCHS} epochs... ###")
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=EPOCHS,
    initial_epoch=initial_epoch,
    callbacks=[schedule_callback, metrics_callback]
)

## Single Video Prediction

### Overview
This section demonstrates **inference on a single video** using the trained LipNet model, loading weights from the latest checkpoint and comparing predictions with ground truth text.

---

### 1. Enhanced Video Loading
```python
def load_video_with_progress(path: str):
    cap = cv2.VideoCapture(path)
    frames = []
    for _ in tqdm(range(int(cap.get(cv2.CAP_PROP_FRAME_COUNT))), desc="Processing Video Frames"):
        ret, frame = cap.read()
        if not ret: break
        frame = tf.image.rgb_to_grayscale(frame)
        frames.append(frame[190:236, 80:220, :])
```
- **Progress tracking**: Uses TQDM to show frame processing progress
- **Same preprocessing**: Applies identical cropping and normalization as training
- **Error handling**: Returns empty array if video loading fails

### 2. Model Loading
```python
predictor_model = build_model()
latest_checkpoint_path = max(checkpoint_files, key=os.path.getctime)
predictor_model.load_weights(latest_checkpoint_path)
```
- **Fresh model instance**: Rebuilds architecture for inference
- **Latest checkpoint**: Automatically selects most recent saved weights
- **Weight restoration**: Loads trained parameters for prediction

### 3. Test Video Selection
```python
all_files = sorted(glob.glob(os.path.join(GDRIVE_DATA_PATH, '*.mpg')))
split_point = int(len(all_files) * 0.9)
test_video_list = all_files[split_point:]
video_path = random.choice(test_video_list)
```
- **Consistent splitting**: Uses same 90/10 split as training
- **Test set selection**: Chooses from validation portion of dataset
- **Random sampling**: Selects a random test video for demonstration

### 4. Prediction Process
```python
yhat = predictor_model.predict(video_batch, verbose=1)
decoded_preds = tf.keras.backend.ctc_decode(yhat, input_length, greedy=True)[0][0].numpy()
```
- **Batch preparation**: Expands single video to batch format
- **Model inference**: Generates character probability predictions
- **CTC decoding**: Converts probabilities to readable text using greedy search

### 5. Results Comparison
```python
original_text = tf.strings.reduce_join(num_to_char(original_tokens)).numpy().decode('utf-8')
predicted_text = tf.strings.reduce_join(num_to_char(decoded_preds[0])).numpy().decode('utf-8')
```
- **Ground truth loading**: Reads original alignment file
- **Text conversion**: Converts both original and predicted tokens to strings
- **Side-by-side display**: Shows original vs predicted text for evaluation

### 6. Visualization
- **Sample frame**: Displays middle frame from the processed video
- **Visual context**: Shows the lip region that the model analyzed
- **Grayscale rendering**: Matches the preprocessing used during training

In [ ]:
### 10. Predict on a Single Video (Reading Directly from Google Drive)

import random
from tqdm.notebook import tqdm

print("### Setting up for prediction...")

def load_video_with_progress(path:str):
    """Loads and preprocesses video frames with a TQDM progress bar."""
    cap = cv2.VideoCapture(path)
    frames = []
    for _ in tqdm(range(int(cap.get(cv2.CAP_PROP_FRAME_COUNT))), desc="Processing Video Frames"):
        ret, frame = cap.read()
        if not ret: break
        frame = tf.image.rgb_to_grayscale(frame)
        frames.append(frame[190:236, 80:220, :])
    cap.release()

    if not frames: return np.array([])
    mean = tf.math.reduce_mean(frames)
    std = tf.math.reduce_std(tf.cast(frames, tf.float32))
    return tf.cast((frames - mean), tf.float32) / std

# --- 1. Re-build and Load the Trained Model ---
predictor_model = build_model()
checkpoint_files = glob.glob(os.path.join(CHECKPOINT_DIR, "epoch_*.weights.h5"))
if not checkpoint_files:
    raise Exception("No checkpoint files found. Please train the model first.")

latest_checkpoint_path = max(checkpoint_files, key=os.path.getctime)
print(f"Loading weights from: {latest_checkpoint_path}")
predictor_model.load_weights(latest_checkpoint_path)
print("Model weights loaded successfully.")

# --- 2. Select a Video to Predict (from Google Drive) ---
all_files = sorted(glob.glob(os.path.join(GDRIVE_DATA_PATH, '*.mpg')))

if not all_files:
    raise FileNotFoundError(f"ERROR: No .mpg files found in your Google Drive path: '{GDRIVE_DATA_PATH}'.")

split_point = int(len(all_files) * 0.9)
test_video_list = all_files[split_point:]

if not test_video_list:
    raise ValueError("ERROR: The test video list is empty. Your dataset may be too small for a 90/10 split.")

video_path = random.choice(test_video_list)
print(f"Selected video for prediction: {video_path}")

# --- 3. Load and Preprocess the Video ---
video = load_video_with_progress(video_path)
video_batch = tf.expand_dims(video, axis=0)

# --- 4. Make a Prediction ---
print("\nRunning model prediction...")
yhat = predictor_model.predict(video_batch, verbose=1)
input_length = [yhat.shape[1]] * yhat.shape[0]
decoded_preds = tf.keras.backend.ctc_decode(yhat, input_length, greedy=True)[0][0].numpy()

# --- 5. Get Original and Predicted Text ---
base_dir = os.path.dirname(video_path)
file_name = os.path.basename(video_path).split('.')[0]
alignment_path = os.path.join(base_dir, 'align', f'{file_name}.align')

original_tokens = load_alignments(alignment_path)
original_text = tf.strings.reduce_join(num_to_char(original_tokens)).numpy().decode('utf-8')
predicted_text = tf.strings.reduce_join(num_to_char(decoded_preds[0])).numpy().decode('utf-8')

# --- 6. Display the Results ---
print("\n--- PREDICTION COMPLETE ---")
print(f"\nORIGINAL:   {original_text}")
print(f"PREDICTED:  {predicted_text}")
print("---------------------------\n")

sample_frame = video[len(video) // 2]
plt.figure(figsize=(8, 4))
plt.imshow(tf.squeeze(sample_frame), cmap='gray')
plt.title("Sample Frame from Video")
plt.axis('off')
plt.show()